# LR analysis: patient-stratified sensitivity check (read-only)
Run the cell below. Set DATA_FOLDER if your path differs. Uses the saved pseudobulk object; nothing upstream is re-run.

In [1]:
"""
LR analysis -- patient-stratified sensitivity check (read-only).

Re-tests every LR pair with patient taken into account, using the saved
pseudobulk object (one score per region = one nerve or one tumour region per
section). Nothing upstream is re-run and no existing table or figure is
changed: results go to a separate 'sensitivity' folder.

Tests per pair, for InvN vs NI and InvN vs Tumour:
  1. van Elteren test (stratified Wilcoxon / Mann-Whitney): regions are
     ranked within each patient only, the per-patient rank sums are combined
     (weights 1/(N_k+1)). Two-sided, so directly comparable with the original
     pooled Mann-Whitney p values.
  2. Stouffer combination of per-patient one-sided Mann-Whitney tests
     (InvN > reference), weighted by sqrt(regions in that patient). Opposite
     directions in different patients cancel out.
  BH correction across all tested pairs, as in the original analysis.

Also reported per patient: mean difference (InvN - reference) and the
one-sided Mann-Whitney p value.

Outputs (in DATA_FOLDER/sensitivity/):
  LR_stratified_all_pairs.csv        every tested pair, original + stratified
  LR_stratified_Tier1.csv            the 130 Tier 1 pairs, with 'still significant'
  LR_stratified_shown_pairs.csv      the pairs shown in the main figure
  LR_regions_per_patient.csv         number of regions per group x patient

Run from the project folder, after 01_LR_analysis.ipynb.
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"   # network drive, before h5py import
import h5py
import anndata as ad
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, norm, rankdata
from scipy.sparse import issparse
import statsmodels.stats.multitest as mt

# ---- Settings ---------------------------------------------------------------
# Written by 01_LR_analysis.ipynb. Paths are relative to the project folder.
DATA_FOLDER = os.path.join('output', 'data')
PDATA_FILE  = os.path.join(DATA_FOLDER, 'LR_pseudobulk_pdata.h5ad')
MASTER_FILE = os.path.join(DATA_FOLDER, 'SpatialLR_MASTER_TABLE.csv')
OUT_FOLDER  = os.path.join(DATA_FOLDER, 'sensitivity')
os.makedirs(OUT_FOLDER, exist_ok=True)

ALPHA = 0.05
PATIENTS = ['54703', '6249', '70865']
PATIENT_LABELS = {'54703': 'P1', '6249': 'P2', '70865': 'P3'}
INV, REFS = 'Invaded', {'NI': 'NonInvaded', 'Tumor': 'Tumor'}

# Pairs shown in the main figure (same as grid_rows in the LR notebook)
SHOWN_PAIRS = ['GDNF^GFRA3', 'NRG1^ERBB2_ERBB3', 'NTF4^NGFR', 'SEMA3B^NRP2_PLXNA1', 'COL9A3^SDC1',
               'COL4A1^ITGAV_ITGB8', 'TNC^ITGAV_ITGB3', 'SPP1^ITGAV_ITGB3', 'TGM2^TBXA2R', 'LAMA2^ITGA6_ITGB4']

# ---- Data -------------------------------------------------------------------
with h5py.File(PDATA_FILE, "r", locking=False) as _h5:
    _read_elem = getattr(getattr(ad, "io", None), "read_elem", None) or ad.experimental.read_elem
    pdata = _read_elem(_h5)
obs = pdata.obs.copy()
obs['patient_id'] = obs['patient_id'].astype(str)
obs['Group'] = obs['Group'].astype(str)
X = pdata.X.toarray() if issparse(pdata.X) else np.asarray(pdata.X)
pairs = np.asarray(pdata.var_names).astype(str)
print("Pseudobulk regions x pairs:", X.shape)

master = pd.read_csv(MASTER_FILE)
print("Master table:", master.shape)

counts = pd.crosstab(obs['Group'], obs['patient_id']).reindex(columns=PATIENTS)
counts.columns = [PATIENT_LABELS[p] for p in counts.columns]
counts.to_csv(os.path.join(OUT_FOLDER, 'LR_regions_per_patient.csv'))
print("\nRegions per group x patient:\n", counts)

grp = obs['Group'].to_numpy()
pat = obs['patient_id'].to_numpy()


# ---- Tests ------------------------------------------------------------------
def van_elteren(x_by_pat, y_by_pat):
    """Two-sided van Elteren test. x = InvN, y = reference, lists per patient."""
    T, V = 0.0, 0.0
    for x, y in zip(x_by_pat, y_by_pat):
        n1, n2 = len(x), len(y)
        if n1 == 0 or n2 == 0:
            continue
        N = n1 + n2
        r = rankdata(np.concatenate([x, y]))
        W = r[:n1].sum()
        _, t = np.unique(r, return_counts=True)
        tie = (t ** 3 - t).sum() / (N * (N - 1)) if N > 1 else 0.0
        var = n1 * n2 / 12.0 * ((N + 1) - tie)
        w = 1.0 / (N + 1)
        T += w * (W - n1 * (N + 1) / 2.0)
        V += w ** 2 * var
    if V <= 0:
        return np.nan, np.nan
    z = T / np.sqrt(V)
    return z, 2 * norm.sf(abs(z))


def stouffer(x_by_pat, y_by_pat):
    """Per-patient one-sided MWU (x > y), combined by weighted Stouffer.
    Returns combined z, two-sided p (for comparability), per-patient one-sided p."""
    zs, ws, ps = [], [], []
    for x, y in zip(x_by_pat, y_by_pat):
        if len(x) == 0 or len(y) == 0 or np.ptp(np.concatenate([x, y])) == 0:
            ps.append(np.nan)
            continue
        p = mannwhitneyu(x, y, alternative='greater').pvalue
        p = min(max(p, 1e-300), 1 - 1e-16)
        ps.append(p)
        zs.append(norm.isf(p))
        ws.append(np.sqrt(len(x) + len(y)))
    if not zs:
        return np.nan, np.nan, ps
    zs, ws = np.array(zs), np.array(ws)
    z = (ws * zs).sum() / np.sqrt((ws ** 2).sum())
    return z, 2 * norm.sf(abs(z)), ps


rows = []
for j, pair in enumerate(pairs):
    v = X[:, j]
    row = {'Interaction': pair}
    for lab, ref in REFS.items():
        xs = [v[(grp == INV) & (pat == p)] for p in PATIENTS]
        ys = [v[(grp == ref) & (pat == p)] for p in PATIENTS]
        z_ve, p_ve = van_elteren(xs, ys)
        z_st, p_st, p_pp = stouffer(xs, ys)
        row[f'vanElteren_z_vs_{lab}'] = z_ve
        row[f'vanElteren_p_vs_{lab}'] = p_ve
        row[f'Stouffer_z_vs_{lab}'] = z_st
        row[f'Stouffer_p_vs_{lab}'] = p_st
        for p, x, y, pp in zip(PATIENTS, xs, ys, p_pp):
            P = PATIENT_LABELS[p]
            row[f'diff_{P}_vs_{lab}'] = (x.mean() - y.mean()) if len(x) and len(y) else np.nan
            row[f'MWU_onesided_p_{P}_vs_{lab}'] = pp
        diffs = [row[f'diff_{PATIENT_LABELS[p]}_vs_{lab}'] for p in PATIENTS]
        row[f'n_patients_higher_vs_{lab}'] = int(np.nansum(np.array(diffs) > 0))
    rows.append(row)
strat = pd.DataFrame(rows)

# BH across all tested pairs (same family as the original analysis)
tested = strat['Interaction'].isin(master['Interaction'])
print(f"\nPairs in pseudobulk: {len(strat)}; in master table (tested): {tested.sum()}")
strat = strat[tested].copy()
for lab in REFS:
    for test in ['vanElteren', 'Stouffer']:
        strat[f'{test}_padj_vs_{lab}'] = mt.multipletests(strat[f'{test}_p_vs_{lab}'].fillna(1),
                                                          method='fdr_bh')[1]

keep = ['Interaction', 'KW_padj', 'InvVsNI_padj', 'InvVsTumor_padj',
        'mean_Invaded', 'mean_NonInvaded', 'mean_Tumor', 'n_Invaded', 'n_NonInvaded', 'n_Tumor']
allp = master[[c for c in keep if c in master.columns]].merge(strat, on='Interaction', how='left')

# Original Tier 1 definition (as in the notebook)
tier1 = ((allp['KW_padj'] < ALPHA) & (allp['InvVsNI_padj'] < ALPHA) & (allp['InvVsTumor_padj'] < ALPHA) &
         (allp['mean_Invaded'] > allp['mean_NonInvaded']) & (allp['mean_Invaded'] > allp['mean_Tumor']))
allp['Tier1_original'] = tier1
for test in ['vanElteren', 'Stouffer']:
    allp[f'Tier1_still_sig_{test}'] = tier1 & (allp[f'{test}_padj_vs_NI'] < ALPHA) & \
                                      (allp[f'{test}_padj_vs_Tumor'] < ALPHA)
    allp[f'Tier1_if_{test}'] = (allp['KW_padj'] < ALPHA) & (allp[f'{test}_padj_vs_NI'] < ALPHA) & \
                               (allp[f'{test}_padj_vs_Tumor'] < ALPHA) & \
                               (allp['mean_Invaded'] > allp['mean_NonInvaded']) & \
                               (allp['mean_Invaded'] > allp['mean_Tumor'])
allp['all3_patients_higher'] = (allp['n_patients_higher_vs_NI'] == 3) & (allp['n_patients_higher_vs_Tumor'] == 3)

allp.to_csv(os.path.join(OUT_FOLDER, 'LR_stratified_all_pairs.csv'), index=False)
t1 = allp[allp['Tier1_original']].sort_values('InvVsNI_padj')
t1.to_csv(os.path.join(OUT_FOLDER, 'LR_stratified_Tier1.csv'), index=False)

# ---- Summary ----------------------------------------------------------------
n1 = len(t1)
print(f"\n=== Tier 1 (original, pooled Mann-Whitney): {n1} pairs ===")
for test in ['vanElteren', 'Stouffer']:
    print(f"Still significant vs NI and vs Tumour with {test:10s}: "
          f"{int(t1[f'Tier1_still_sig_{test}'].sum())} / {n1}   "
          f"(Tier 1 if defined with {test}: {int(allp[f'Tier1_if_{test}'].sum())} pairs)")
print(f"Higher in InvN in all 3 patients (vs NI and vs Tumour): {int(t1['all3_patients_higher'].sum())} / {n1}")

shown_cols = ['Interaction', 'InvVsNI_padj', 'vanElteren_padj_vs_NI', 'Stouffer_padj_vs_NI',
              'InvVsTumor_padj', 'vanElteren_padj_vs_Tumor', 'Stouffer_padj_vs_Tumor',
              'n_patients_higher_vs_NI', 'n_patients_higher_vs_Tumor'] + \
             [f'diff_{PATIENT_LABELS[p]}_vs_{lab}' for lab in REFS for p in PATIENTS] + \
             [f'MWU_onesided_p_{PATIENT_LABELS[p]}_vs_{lab}' for lab in REFS for p in PATIENTS]
shown = allp[allp['Interaction'].isin(SHOWN_PAIRS)].set_index('Interaction').reindex(SHOWN_PAIRS).reset_index()
missing = shown.loc[shown['InvVsNI_padj'].isna(), 'Interaction'].tolist()
if missing:
    print("\nWARNING: not found in the master table:", missing)
shown[shown_cols].to_csv(os.path.join(OUT_FOLDER, 'LR_stratified_shown_pairs.csv'), index=False)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
print("\n=== Pairs shown in the figure: BH-adjusted p, pooled vs stratified ===")
fmt = shown[['Interaction', 'InvVsNI_padj', 'vanElteren_padj_vs_NI', 'Stouffer_padj_vs_NI',
             'InvVsTumor_padj', 'vanElteren_padj_vs_Tumor', 'Stouffer_padj_vs_Tumor',
             'n_patients_higher_vs_NI', 'n_patients_higher_vs_Tumor']].copy()
fmt.columns = ['Pair', 'NI pooled', 'NI vanEl', 'NI Stouf', 'Tum pooled', 'Tum vanEl', 'Tum Stouf',
               'pts>NI', 'pts>Tum']
print(fmt.to_string(index=False, float_format=lambda x: f"{x:.2g}"))
print("\nPer-patient differences (InvN - reference) for the shown pairs:")
print(shown[['Interaction'] + [f'diff_{PATIENT_LABELS[p]}_vs_{lab}' for lab in REFS for p in PATIENTS]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nWritten to:", OUT_FOLDER)


Pseudobulk regions x pairs: (176, 4372)
Master table: (4366, 18)

Regions per group x patient:
             P1  P2  P3
Group                 
Invaded     15  17  15
NonInvaded  13  16  12
Tumor       28  21  33
none         2   2   2

Pairs in pseudobulk: 4372; in master table (tested): 4366

=== Tier 1 (original, pooled Mann-Whitney): 130 pairs ===
Still significant vs NI and vs Tumour with vanElteren: 112 / 130   (Tier 1 if defined with vanElteren: 133 pairs)
Still significant vs NI and vs Tumour with Stouffer  : 109 / 130   (Tier 1 if defined with Stouffer: 126 pairs)
Higher in InvN in all 3 patients (vs NI and vs Tumour): 89 / 130

=== Pairs shown in the figure: BH-adjusted p, pooled vs stratified ===
              Pair  NI pooled  NI vanEl  NI Stouf  Tum pooled  Tum vanEl  Tum Stouf  pts>NI  pts>Tum
        GDNF^GFRA3      0.028     0.024     0.024       2e-10    3.1e-10      3e-10       3        3
  NRG1^ERBB2_ERBB3    2.3e-05   1.2e-05   1.3e-05       0.044      0.049      0.052